# 2. Modelling — version 2
Same inputs, artifacts and checkpoint bundle as version 1, so `prediction_code_version1.ipynb`
runs unchanged against it. Requires a data snapshot from `chunking_code_version2.ipynb`
(needs `IS_BHAIDOOJ` and `IS_DIWALI_WINDOW` in the calendar).

**Why version 1 was slow.** Darts builds every training sample by calling `series[i]` for a random
series. Version 1 answered that with `np.load` + `DatetimeIndex` + masking + a new `TimeSeries`
on every call; its `lru_cache(maxsize=128)` holds 128 of ~104,000 series, so it almost never hit.
The GPU was waiting on file reads. Version 0 kept each built series in a RAM dict after first use.

**What changed vs version 1**
1. **Data path:** all target history is loaded once into one in-RAM matrix; each series' `TimeSeries`
   is built once and kept in a dict (version 0 pattern). No disk reads during training.
   The per-series `.npz` cache is still written, only because the prediction notebook reads it.
2. **Sample weights** are built here from `PENALTY_COLS`. `FESTIVE_DAYS_FROM_DIWALI` is clipped to
   [-20, +10] in the source, so "non-zero" is true on almost every day of the year; it is
   represented by `IS_DIWALI_WINDOW` (true calendar offsets, built in chunking) instead.
3. **Trainer settings back to version 0:** batch 256, no gradient accumulation, 3000 batches/epoch.
   Change them in the settings cell; they are recorded in `training_settings.json`.
4. **Epoch timer:** after the first 100 batches it prints projected minutes for the epoch and
   logs `epoch_minutes` / `val_minutes` to TensorBoard.

In [ ]:
import smtplib
from email.mime.text import MIMEText
from dotenv import load_dotenv
import os

load_dotenv()
my_password = os.getenv('EMAIL_APP_PASSWORD')

def send_notification(subject, body):
    msg = MIMEText(body)
    msg['Subject'] = subject
    msg['From'] = "yash.saxenaust.ext@heromotocorp.com"
    msg['To'] = "yash.saxena@ust.com"

    with smtplib.SMTP("smtp.gmail.com", 587) as server:
        server.starttls()
        server.login("yash.saxenaust.ext@heromotocorp.com", my_password)
        server.send_message(msg)

In [ ]:
from pathlib import Path
import json, hashlib, uuid, platform
from datetime import datetime, timezone
import numpy as np
import pandas as pd


def digest(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(8 * 1024 * 1024), b""):
            h.update(b)
    return h.hexdigest()


def write_json(path, obj):
    path = Path(path)
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(json.dumps(obj, indent=2, allow_nan=False), encoding="utf-8")
    tmp.replace(path)


def read_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))


def check_daily(frame, date_col, start, end, label):
    idx = pd.DatetimeIndex(pd.to_datetime(frame[date_col]))
    expected = pd.date_range(start, end, freq="D")
    if idx.has_duplicates or not idx.equals(expected):
        raise ValueError(
            f"{label}: duplicate, missing or unexpected dates; expected {len(expected)} daily rows, got {len(idx)}. Repair the source grid; missing observations are not assumed to be zero."
        )


SEGMENTS = {
    "SPLENDOR+": "100 CC",
    "HF DELUXE": "100 CC",
    "HF 100": "100 CC",
    "PASSION": "100 CC",
    "GLAMOUR": "125 CC",
    "SUPER SPLENDOR": "125 CC",
    "XTREME 125": "125 CC",
    "XPULSE": "PREMIUM",
    "XTREME 160": "PREMIUM",
    "XTREME 250": "PREMIUM",
    "DESTINI": "SCOOTER",
    "PLEASURE+": "SCOOTER",
    "XOOM": "SCOOTER",
}

In [ ]:
PROJECT_DIR = Path.cwd()
RUN_DIR_OVERRIDE = None
RUN_DIR = Path(
    RUN_DIR_OVERRIDE or read_json(PROJECT_DIR / "iteration3_active_run.json")["run_dir"]
)
if not (RUN_DIR / "DATA_READY").exists():
    raise RuntimeError("Data snapshot is incomplete")
cfg = read_json(RUN_DIR / "config.json")
dm = read_json(RUN_DIR / "data_manifest.json")
for name, expected in [
    ("config.json", dm["config_hash"]),
    ("calendar.parquet", dm["calendar_hash"]),
    ("selected_series.parquet", dm["membership_hash"]),
]:
    if digest(RUN_DIR / name) != expected:
        raise RuntimeError(f"Data snapshot changed: {name}; create a new run")
TIME, KEY, TARGET = (cfg["time_col"], cfg["group_col"], cfg["target_col"])
STATIC, FUTURE = (cfg["static_covariates"], cfg["future_covariates"])
ICL, OCL = (cfg["icl"], cfg["ocl"])
TRAIN_START, TRAIN_END = (
    pd.Timestamp(cfg["train_start"]),
    pd.Timestamp(cfg["train_end"]),
)
HISTORY_END = pd.Timestamp(cfg["val_end"])
FC_START, FC_END = (
    pd.Timestamp(cfg["forecast_start"]),
    pd.Timestamp(cfg["forecast_end"]),
)
VAL_OUTPUT_START = pd.Timestamp(cfg["val_output_start"])
VAL_INPUT_START = VAL_OUTPUT_START - pd.Timedelta(days=ICL)
assert HISTORY_END + pd.Timedelta(days=1) == FC_START
assert OCL == (FC_END - FC_START).days + 1
assert VAL_OUTPUT_START > TRAIN_END
print("Data snapshot:", RUN_DIR, "| Population:", cfg["population"])

for need in ['IS_BHAIDOOJ', 'IS_DIWALI_WINDOW']:
    if need not in FUTURE:
        raise RuntimeError(f'{need} missing from this snapshot. Run chunking_code_version2.ipynb first.')

In [ ]:
import torch, darts, inspect, pickle, gc
from darts import TimeSeries
from darts.models import TFTModel
from darts.utils.likelihood_models import NegativeBinomialLikelihood
import pytorch_lightning as pl

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA GPU not visible. Select your CUDA-enabled PyTorch environment/kernel. This notebook will not silently fall back to CPU."
    )
torch.set_float32_matmul_precision("high")
PRECISION = "bf16-mixed" if torch.cuda.is_bf16_supported() else "32-true"
print("GPU:", torch.cuda.get_device_name(0))
print(
    "VRAM (GiB):", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2)
)
print(
    "Darts:",
    darts.__version__,
    "| PyTorch:",
    torch.__version__,
    "| precision:",
    PRECISION,
)

In [ ]:
import time
from darts.dataprocessing.transformers import StaticCovariatesTransformer
from sklearn.preprocessing import OrdinalEncoder
from pytorch_lightning.callbacks import EarlyStopping
from pytorch_lightning.loggers import CSVLogger, TensorBoardLogger

# ---- training settings (version 0 values) -------------------------------------------------
BATCH_SIZE = 256
ACCUMULATE_GRAD_BATCHES = 1
MAX_EPOCHS = 100
MAX_TRAIN_BATCHES = 3000          # batches per "epoch"; None = full pass (~68.8M windows)
LIMIT_VAL_BATCHES = 1.0           # 1 window per series; 1.0 = every series
HIDDEN_SIZE, LSTM_LAYERS, HEADS = (32, 4, 4)
DROPOUT = 0.05
NUM_WORKERS = 0                   # Windows + Jupyter: >0 deadlocks (spawn). Keep 0.

# ---- RAM data path ------------------------------------------------------------------------
CACHE_VAL_SERIES = True           # ~1 GiB for 104k series; False builds val series from RAM each pass
WARM_CACHE = True                 # build every series before fit so epoch 1 timing is honest
TIMER_PROBE_BATCHES = 100         # projected epoch time printed after this many batches

# ---- festive sample weights ---------------------------------------------------------------
PENALTY_COLS = ['FESTIVE_DAYS_FROM_DIWALI', 'IS_NAVRATARI', 'IS_NAVRATARI_START',
                'IS_NAVRATARI_END', 'IS_DUSSEHRA', 'IS_PITRA_PAKSHA', 'IS_DHANTERAS',
                'IS_BHAIDOOJ']
FESTIVE_MULTIPLIER = 4.0          # weight on penalty days; 1.0 = off. Version 0 used 1 + 3 = 4.

if "sample_weight" not in inspect.signature(TFTModel.fit).parameters:
    raise RuntimeError("This Darts version lacks sample_weight. Use a compatible environment; do not drop weights.")
ATTEMPT = datetime.now(timezone.utc).strftime("fit_%Y%m%d_%H%M%S_") + uuid.uuid4().hex[:8]
FIT_DIR = RUN_DIR / ATTEMPT
FIT_DIR.mkdir(exist_ok=False)
CACHE_DIR = FIT_DIR / "series_cache"
CACHE_DIR.mkdir()
MODEL_NAME = "daily_tft_" + ATTEMPT
WORK_DIR = FIT_DIR / "darts_logs"
WORK_DIR.mkdir()
CSV_LOGGER = CSVLogger(save_dir=str(FIT_DIR), name="training_metrics")
TENSORBOARD_LOGGER = TensorBoardLogger(save_dir=str(FIT_DIR), name="tensorboard")
print("TensorBoard log directory:", TENSORBOARD_LOGGER.log_dir)
print(f'Launch with: tensorboard --logdir "{FIT_DIR / "tensorboard"}"')

In [ ]:
calendar = pd.read_parquet(RUN_DIR / "calendar.parquet")
calendar[TIME] = pd.to_datetime(calendar[TIME])
check_daily(calendar, TIME, TRAIN_START, FC_END, "Calendar")
if "WEIGHT" in calendar.columns:
    print("Ignoring WEIGHT column from the snapshot; weights are rebuilt below from PENALTY_COLS.")
    calendar = calendar.drop(columns="WEIGHT")

missing = sorted(set(PENALTY_COLS) - set(calendar.columns))
if missing:
    raise ValueError(f"Penalty columns missing from calendar: {missing}")

# FESTIVE_DAYS_FROM_DIWALI is clipped to [-20, +10] (then /20) in the source, so it is non-zero on
# almost every day: 60 days before Diwali also reads -20. It must never be used as "!= 0".
# IS_DIWALI_WINDOW was built in chunking from each year's actual Diwali date: 1 only on days
# -20..+10, so it is the correct stand-in for that column.
flag_cols = [c for c in PENALTY_COLS if c != "FESTIVE_DAYS_FROM_DIWALI"]
if not calendar[flag_cols].isin([0, 1]).all().all():
    raise ValueError("Penalty flags must be 0/1")
diwali = calendar["IS_DIWALI_WINDOW"].eq(1)
if "FESTIVE_DAYS_FROM_DIWALI" not in PENALTY_COLS:
    diwali[:] = False

# Cross-check the window against the (unclipped) interior of the distance feature:
# a scaled value strictly between -1 and +0.5 is an unambiguous -19..+9 day and must be inside.
dist = calendar["FESTIVE_DAYS_FROM_DIWALI"].to_numpy() * 20.0
interior = (dist > -20 + 1e-6) & (dist < 10 - 1e-6)
if not calendar.loc[interior, "IS_DIWALI_WINDOW"].eq(1).all():
    raise ValueError("IS_DIWALI_WINDOW disagrees with FESTIVE_DAYS_FROM_DIWALI; rebuild the snapshot")
per_year = calendar.groupby(calendar[TIME].dt.year)["IS_DIWALI_WINDOW"].sum().astype(int)
if (per_year != 31).any():
    raise ValueError(f"Expected a 31-day Diwali window per year, got {per_year.to_dict()}")

penalty_day = diwali | calendar[flag_cols].eq(1).any(axis=1)
calendar["WEIGHT"] = np.where(penalty_day, FESTIVE_MULTIPLIER, 1.0).astype(np.float32)

naive = (calendar[PENALTY_COLS] != 0).any(axis=1)
year = calendar[TIME].dt.year
report = pd.DataFrame({"penalty_days": penalty_day.groupby(year).sum(),
                       "of_which_diwali_window": diwali.groupby(year).sum(),
                       "naive_nonzero_rule_would_flag": naive.groupby(year).sum(),
                       "days_in_calendar": year.groupby(year).size()})
print(report)
print(f"Share of calendar weighted: {penalty_day.mean():.1%} "
      f"(the naive '!= 0' rule would weight {naive.mean():.1%})")

SHARED_COV = TimeSeries.from_dataframe(
    calendar, time_col=TIME, value_cols=FUTURE, freq="D", fill_missing_dates=False
).astype(np.float32)
SHARED_WEIGHT = TimeSeries.from_dataframe(
    calendar, time_col=TIME, value_cols=["WEIGHT"], freq="D", fill_missing_dates=False
).astype(np.float32)
calendar[[TIME, "WEIGHT"]].to_parquet(FIT_DIR / "sample_weight.parquet", index=False)

In [ ]:
# Same validation as version 1. New: every series' history also goes into one RAM matrix
# (SALES), so training never touches disk. The .npz files are kept for the prediction notebook.
selected = pd.read_parquet(RUN_DIR / "selected_series.parquet")[KEY].astype(str)
expected_end = FC_END if cfg["mode"] == "festive_backtest_2025" else HISTORY_END
HIST_TIMES = pd.date_range(TRAIN_START, HISTORY_END, freq="D")
SALES = np.empty((len(selected), len(HIST_TIMES)), dtype=np.float32)
print(f"RAM target matrix: {SALES.shape[0]:,} series x {SALES.shape[1]:,} days "
      f"= {SALES.nbytes / 2**30:.2f} GiB")

records, static_rows, seen, audit = ([], [], set(), [])
t0 = time.perf_counter()
for entry in dm["chunks"]:
    path = RUN_DIR / entry["path"]
    if digest(path) != entry["sha256"]:
        raise RuntimeError(f"Chunk changed: {path}")
    frame = pd.read_parquet(path)
    frame[TIME] = pd.to_datetime(frame[TIME])
    for key, g in frame.groupby(KEY, sort=False):
        key = str(key)
        if key in seen:
            raise ValueError(f"Series appears in multiple chunks: {key}")
        seen.add(key)
        g = g.sort_values(TIME)
        check_daily(g, TIME, TRAIN_START, expected_end, key)
        if g[STATIC].isna().any().any() or (g[STATIC].nunique(dropna=False) > 1).any():
            raise ValueError(f"{key}: static attributes change or are missing")
        h = g.loc[g[TIME] <= HISTORY_END]
        values = h[TARGET].to_numpy(dtype=np.float32)
        if (not np.isfinite(values).all() or (values < 0).any()
                or (not np.equal(values, np.rint(values)).all())):
            raise ValueError(f"{key}: invalid count target")
        train = h.loc[h[TIME] <= TRAIN_END, TARGET]
        if len(train) < ICL + OCL:
            raise ValueError(f"{key}: insufficient training history; needs a separate short-history treatment")
        row = len(records)
        if row >= len(SALES):
            raise ValueError("More series in chunks than in selected_series.parquet")
        SALES[row] = values
        name = hashlib.sha256(key.encode("utf-8")).hexdigest() + ".npz"
        np.savez(CACHE_DIR / name, dates=h[TIME].to_numpy(dtype="datetime64[ns]"), sales=values)
        records.append({"key": key, "file": name, "sha256": digest(CACHE_DIR / name)})
        static_rows.append(g[STATIC].iloc[0].astype(str).to_dict())
        audit.append({KEY: key, "MODEL_FAMILY": str(g.MODEL_FAMILY.iloc[0]),
                      "TRAIN_SALES": float(train.sum()),
                      "NEVER_SOLD_IN_TRAIN": bool(train.sum() == 0),
                      "HISTORY_SALES": float(values.sum())})
    del frame
    gc.collect()
    print(f"Validated/cached series: {len(records):,} | {time.perf_counter() - t0:,.0f}s")
if seen != set(selected):
    raise ValueError(f"Membership mismatch: {len(set(selected) - seen)} selected IDs lack source history")
if not records or len(records) != len(SALES):
    raise ValueError("Series count does not match the RAM matrix")
N_TRAIN = int((HIST_TIMES <= TRAIN_END).sum())
VAL_START_IDX = int(HIST_TIMES.get_loc(VAL_INPUT_START))
static_df = pd.DataFrame(static_rows, columns=STATIC)
audit_df = pd.DataFrame(audit)
audit_df.to_csv(FIT_DIR / "series_coverage_audit.csv", index=False)
print(audit_df.groupby("MODEL_FAMILY")[["TRAIN_SALES", "NEVER_SOLD_IN_TRAIN"]].sum())
write_json(FIT_DIR / "cache_manifest.json", records)

In [ ]:
fit_size = int(static_df.nunique().max())
fit_frame = pd.DataFrame(
    {
        c: pd.Series(sorted(static_df[c].unique())).reindex(range(fit_size)).ffill()
        for c in STATIC
    }
)
dummy_dates = pd.date_range("2000-01-01", periods=2)
dummy = [
    TimeSeries.from_times_and_values(
        dummy_dates,
        np.zeros((2, 1), np.float32),
        columns=[TARGET],
        static_covariates=fit_frame.iloc[[i]].reset_index(drop=True),
    )
    for i in range(fit_size)
]
transformer = StaticCovariatesTransformer(
    transformer_cat=OrdinalEncoder(handle_unknown="error"), cols_cat=STATIC
)
transformer.fit(dummy)
encoded = []
for i in range(len(static_df)):
    ts = TimeSeries.from_times_and_values(
        dummy_dates,
        np.zeros((2, 1), np.float32),
        columns=[TARGET],
        static_covariates=static_df.iloc[[i]].reset_index(drop=True),
    )
    encoded.append(transformer.transform(ts).static_covariates.astype(np.float32))
counts = {c: int(static_df[c].nunique()) for c in STATIC}
embeddings = {c: (counts[c], min(50, (counts[c] + 1) // 2)) for c in STATIC}
for stat in encoded:
    for c in STATIC:
        v = float(stat[c].iloc[0])
        if not v.is_integer() or not 0 <= v < counts[c]:
            raise ValueError("Invalid embedding index")
static_df.insert(0, KEY, [r["key"] for r in records])
static_df.to_parquet(FIT_DIR / "static_raw.parquet", index=False)
with open(FIT_DIR / "static_transformer.pkl", "wb") as f:
    pickle.dump(transformer, f)
with open(FIT_DIR / "static_encoded.pkl", "wb") as f:
    pickle.dump(encoded, f)
SHARED_COV.to_pickle(FIT_DIR / "shared_cov.pkl")
del dummy, static_rows
gc.collect()

In [ ]:
from collections.abc import Sequence


class RamTargets(Sequence):
    """Target series served from the in-RAM SALES matrix.

    Each TimeSeries is built once and kept in a dict (version 0 pattern), so every later
    random access by the Darts dataset is a dict lookup. All series share one time index.
    """

    def __init__(self, static_frames, split, cache=True):
        if split == "train":
            self.a, self.b = 0, N_TRAIN
        elif split == "val":
            self.a, self.b = VAL_START_IDX, len(HIST_TIMES)
        else:
            raise ValueError(split)
        self.index = HIST_TIMES[self.a:self.b]
        self.static_frames = static_frames
        self._ram = {} if cache else None

    def __len__(self):
        return len(self.static_frames)

    def __getstate__(self):          # never pickle the cache (checkpoints, workers)
        state = self.__dict__.copy()
        state["_ram"] = {} if self._ram is not None else None
        return state

    def __getitem__(self, i):
        if isinstance(i, slice):
            return [self[j] for j in range(*i.indices(len(self)))]
        if i < 0:
            i += len(self)
        if not 0 <= i < len(self):
            raise IndexError(i)
        if self._ram is not None:
            ts = self._ram.get(i)
            if ts is not None:
                return ts
        ts = TimeSeries.from_times_and_values(
            self.index, SALES[i, self.a:self.b, None], columns=[TARGET],
            static_covariates=self.static_frames[i])
        if self._ram is not None:
            self._ram[i] = ts
        return ts

    def warm(self, label):
        t = time.perf_counter()
        for i in range(len(self)):
            self[i]
            if (i + 1) % 20000 == 0:
                print(f"  {label}: {i + 1:,}/{len(self):,} built")
        print(f"  {label}: {len(self):,} series in {time.perf_counter() - t:,.0f}s")


class Shared(Sequence):

    def __init__(self, ts, n):
        self.ts, self.n = (ts, n)

    def __len__(self):
        return self.n

    def __getitem__(self, i):
        if isinstance(i, slice):
            return [self.ts for _ in range(*i.indices(self.n))]
        if i < 0:
            i += self.n
        if not 0 <= i < self.n:
            raise IndexError(i)
        return self.ts


train_seq = RamTargets(encoded, "train")
val_seq = RamTargets(encoded, "val", cache=CACHE_VAL_SERIES)
train_cov = Shared(SHARED_COV, len(records))
val_cov = Shared(SHARED_COV, len(records))
train_weights = Shared(SHARED_WEIGHT, len(records))
val_weights = Shared(SHARED_WEIGHT, len(records))

# Same series as version 1 would have served, value for value.
for i in [0, len(records) - 1]:
    with np.load(CACHE_DIR / records[i]["file"], allow_pickle=False) as z:
        d, s = pd.DatetimeIndex(z["dates"]), z["sales"]
    assert np.array_equal(train_seq[i].values().ravel(), s[d <= TRAIN_END])
    assert np.array_equal(val_seq[i].values().ravel(), s[(d >= VAL_INPUT_START) & (d <= HISTORY_END)])
assert train_seq[0].end_time() == TRAIN_END
assert val_seq[0].start_time() == VAL_INPUT_START and len(val_seq[0]) == ICL + OCL

if WARM_CACHE:
    print("Building series in RAM (one-time):")
    train_seq.warm("train")
    if CACHE_VAL_SERIES:
        val_seq.warm("val")
try:
    import psutil
    print(f"Process RAM now: {psutil.Process().memory_info().rss / 2**30:.1f} GiB")
except ImportError:
    pass

In [ ]:
class EpochTimer(pl.Callback):
    """Projects epoch time early and logs epoch/validation minutes to TensorBoard."""

    def __init__(self, probe_batches=100):
        self.probe = probe_batches

    def __reduce__(self):
        # Darts pickles trainer callbacks into the saved model. A class defined in this notebook
        # would then be required to load the checkpoint elsewhere (prediction notebook), so it
        # is saved as a plain no-op Lightning callback instead.
        return (pl.Callback, ())

    def on_train_epoch_start(self, trainer, pl_module):
        self.t_epoch = time.perf_counter()

    def on_train_batch_end(self, trainer, pl_module, outputs, batch, batch_idx):
        if batch_idx + 1 == self.probe:
            per = (time.perf_counter() - self.t_epoch) / self.probe
            total = trainer.num_training_batches
            print(f"\n[epoch {trainer.current_epoch}] {per:.3f} s/batch -> projected "
                  f"{per * total / 60:.1f} min for {total:,} batches (validation extra)")

    def on_train_epoch_end(self, trainer, pl_module):
        minutes = (time.perf_counter() - self.t_epoch) / 60
        pl_module.log("epoch_minutes", minutes, on_epoch=True, prog_bar=False)
        print(f"[epoch {trainer.current_epoch}] training took {minutes:.1f} min")

    def on_validation_epoch_start(self, trainer, pl_module):
        self.t_val = time.perf_counter()

    def on_validation_epoch_end(self, trainer, pl_module):
        if trainer.sanity_checking:
            return
        minutes = (time.perf_counter() - self.t_val) / 60
        pl_module.log("val_minutes", minutes, on_epoch=True, prog_bar=False)
        print(f"[epoch {trainer.current_epoch}] validation took {minutes:.1f} min")

In [ ]:
print("Validation outputs:", VAL_OUTPUT_START.date(), "to", HISTORY_END.date())
print(
    "Train weights:",
    np.unique(SHARED_WEIGHT.slice(TRAIN_START, TRAIN_END).values(), return_counts=True),
)
print(
    "Val output weights:",
    np.unique(SHARED_WEIGHT.slice(VAL_OUTPUT_START, HISTORY_END).values(), return_counts=True),
)
total_windows = len(records) * (len(train_seq[0]) - ICL - OCL + 1)
print("Available training windows:", f"{total_windows:,}")
print(
    "Maximum windows per partial epoch:",
    MAX_TRAIN_BATCHES * BATCH_SIZE if MAX_TRAIN_BATCHES else "all",
)
train_settings = dict(
    data_path="in_ram_cached",
    penalty_cols=PENALTY_COLS,
    festive_multiplier=FESTIVE_MULTIPLIER,
    weighted_days_share=float(penalty_day.mean()),
    limit_val_batches=LIMIT_VAL_BATCHES,
    num_workers=NUM_WORKERS,
    batch_size=BATCH_SIZE,
    accumulation=ACCUMULATE_GRAD_BATCHES,
    max_epochs=MAX_EPOCHS,
    max_train_batches=MAX_TRAIN_BATCHES,
    hidden_size=HIDDEN_SIZE,
    lstm_layers=LSTM_LAYERS,
    heads=HEADS,
    dropout=DROPOUT,
    precision=PRECISION,
    csv_log_dir=CSV_LOGGER.log_dir,
    tensorboard_log_dir=TENSORBOARD_LOGGER.log_dir,
)
write_json(FIT_DIR / "training_settings.json", train_settings)
model = TFTModel(
    input_chunk_length=ICL,
    output_chunk_length=OCL,
    hidden_size=HIDDEN_SIZE,
    lstm_layers=LSTM_LAYERS,
    num_attention_heads=HEADS,
    dropout=DROPOUT,
    batch_size=BATCH_SIZE,
    n_epochs=MAX_EPOCHS,
    likelihood=NegativeBinomialLikelihood(),
    loss_fn=None,
    use_reversible_instance_norm=False,
    categorical_embedding_sizes=embeddings,
    random_state=42,
    add_relative_index=True,
    save_checkpoints=True,
    force_reset=False,
    model_name=MODEL_NAME,
    work_dir=str(WORK_DIR),
    pl_trainer_kwargs=dict(
        accelerator="gpu",
        devices=1,
        precision=PRECISION,
        callbacks=[
            EarlyStopping(monitor="val_loss", patience=5, min_delta=0.0001, mode="min"),
            EpochTimer(TIMER_PROBE_BATCHES),
        ],
        gradient_clip_val=0.1,
        accumulate_grad_batches=ACCUMULATE_GRAD_BATCHES,
        limit_train_batches=MAX_TRAIN_BATCHES if MAX_TRAIN_BATCHES else 1.0,
        limit_val_batches=LIMIT_VAL_BATCHES,
        logger=[CSV_LOGGER, TENSORBOARD_LOGGER],
        log_every_n_steps=50,
        enable_progress_bar=True,
    ),
)

try:
    model.fit(
        series=train_seq,
        future_covariates=train_cov,
        val_series=val_seq,
        val_future_covariates=val_cov,
        sample_weight=train_weights,
        val_sample_weight=val_weights,
        dataloader_kwargs={"num_workers": NUM_WORKERS, "pin_memory": True},
        verbose=True,
    )

    send_notification('Training Complete',"""Training is completed""")

except Exception as e:
    send_notification("Forecast FAILED", f"Error: {e}")
    raise 


if model.trainer.strategy.root_device.type != "cuda":
    raise RuntimeError("Training did not use CUDA")
embedding_count = sum((p.numel() for p in model.model.input_embeddings.parameters()))
if embedding_count <= 0:
    raise RuntimeError("Categorical embeddings were not built")
print("GPU training verified. Embedding parameters:", embedding_count)
best = TFTModel.load_from_checkpoint(
    model_name=MODEL_NAME, work_dir=str(WORK_DIR), best=True, map_location="cpu"
)
assert best.input_chunk_length == ICL and best.output_chunk_length == OCL
files = [
    "cache_manifest.json",
    "static_raw.parquet",
    "static_transformer.pkl",
    "static_encoded.pkl",
    "shared_cov.pkl",
    "training_settings.json",
    "sample_weight.parquet",
]
checkpoint_files = list(WORK_DIR.rglob("*"))
files += [
    str(p.relative_to(FIT_DIR))
    for p in checkpoint_files
    if p.is_file() and (p.suffix == ".ckpt" or p.name.endswith(".pth.tar"))
]
bundle = dict(
    run_id=cfg["run_id"],
    model_name=MODEL_NAME,
    work_dir=str(WORK_DIR.resolve()),
    config_hash=dm["config_hash"],
    calendar_hash=dm["calendar_hash"],
    versions={
        "darts": darts.__version__,
        "torch": torch.__version__,
        "lightning": pl.__version__,
    },
    file_hashes={name: digest(FIT_DIR / name) for name in files},
    series_count=len(records),
    gpu=torch.cuda.get_device_name(0),
    embedding_parameters=embedding_count,
    csv_log_dir=CSV_LOGGER.log_dir,
    tensorboard_log_dir=TENSORBOARD_LOGGER.log_dir,
)
write_json(FIT_DIR / "bundle.json", bundle)
write_json(
    RUN_DIR / "active_fit.json",
    {"fit_dir": str(FIT_DIR.resolve()), "bundle_hash": digest(FIT_DIR / "bundle.json")},
)
print("Best checkpoint and matching artifacts ready:", FIT_DIR)
print("Next: prediction_code_version1.ipynb (unchanged)")